In [1]:
import threading
import time

def uji_dining_philosophers(n_filsuf, durasi=1.5):
    garpu = [threading.Lock() for _ in range(n_filsuf)]
    # Arbitrator membatasi maksimal N-1 filsuf yang boleh mengambil garpu
    arbitrator = threading.Semaphore(n_filsuf - 1)
    
    total_makan = [0] * n_filsuf
    berhenti = threading.Event()

    def filsuf(idx):
        kiri = garpu[idx]
        kanan = garpu[(idx + 1) % n_filsuf]
        
        while not berhenti.is_set():
            # Berpikir sebentar
            time.sleep(0.005)
            
            with arbitrator:
                with kiri:
                    with kanan:
                        # Sedang makan
                        total_makan[idx] += 1
                        time.sleep(0.005)

    threads = [
        threading.Thread(target=filsuf, args=(i,)) 
        for i in range(n_filsuf)
    ]

    mulai = time.time()
    for t in threads:
        t.start()

    time.sleep(durasi)
    berhenti.set()

    for t in threads:
        t.join()

    waktu_aktual = time.time() - mulai
    semua_makan = sum(total_makan)
    throughput = semua_makan / waktu_aktual
    
    print(f"N = {n_filsuf:2d} | Total Makan: {semua_makan:4d} "
          f"| Throughput: {throughput:6.2f} makan/detik")

print("=== HASIL PENGUJIAN DINING PHILOSOPHERS ===")
for n in [3, 7, 10]:
    uji_dining_philosophers(n)

=== HASIL PENGUJIAN DINING PHILOSOPHERS ===
N =  3 | Total Makan:  279 | Throughput: 183.90 makan/detik
N =  7 | Total Makan:  557 | Throughput: 365.90 makan/detik
N = 10 | Total Makan: 1030 | Throughput: 675.29 makan/detik


In [ ]:
Berdasarkan hasil eksekusi di atas, nilai throughput (total makan per detik)
meningkat seiring bertambahnya N, tetapi kenaikannya tidak linear 
sempurna (misalnya saat N naik lebih dari 3x lipat dari 3 ke 10, 
throughput tidak langsung naik tepat 3x lipat)